# Single FCM Edge-List Quality Control

## Purpose

This notebook validates one aligned Functional Connectivity Matrix (FCM)
edge-list file from the Curvature-FCN-Aging dataset.

The validation checks the raw file format, data types, rows, nodes, ROI
pairs, missing values, and correlation range.

The final result will be a compact quality-control summary stating whether
the selected FCM file passes validation.



In [1]:
# Import Path for working with file and folder paths
from pathlib import Path

# Import Pandas for loading and analyzing tabular data
import pandas as pd

# Import the existing FCM alignment function
from lemon_connectivity.alignment import align_participants_to_fcm

## Data source

The FCM files are stored in the Curvature-FCN-Aging dataset.
The cohort information file contains the participant IDs used for alignment.

In [2]:
# Define the folder containing the FCM files
fcm_data = Path(
    "../data/external/Curvature-FCN-Aging/DATA/FCM"
)

# Define the cohort metadata file
cohort_path = Path(
    "../data/external/Curvature-FCN-Aging/DATA/cohort_information.tsv"
)

## Load cohort metadata

We load the cohort information with `sub_id` as a string so that
participant identifiers are preserved exactly.

In [3]:
# Load the cohort metadata
cohort_metadata = pd.read_csv(
    cohort_path,
    sep="\t",
    dtype={"sub_id": "string"},
)

## Align participants with FCM files

The existing alignment function checks which cohort participants have
corresponding FCM files.

The alignment result will be used to select the FCM file reproducibly.

In [4]:
# Align cohort participants with the available FCM files
alignment_result = align_participants_to_fcm(
    cohort_metadata,
    fcm_data,)

### exploratory cells 

In [5]:
# Show the column names in the alignment table
alignment_result.alignment_table.columns.tolist()

# Get the unique alignment-status values
alignment_result.alignment_table["alignment_status"].unique()


<StringArray>
['matched']
Length: 1, dtype: string

## 1: Select one FCM file reproducibly

We use the alignment result from the previous step to identify FCM files
that are successfully aligned with cohort participants.

We then sort the valid aligned files alphabetically by their file path
and select the first one.

This makes the file selection deterministic and reproducible.

In [6]:
# Get the alignment table from the previous alignment step
aligned_table = alignment_result.alignment_table

# Keep only records that were successfully matched
valid_aligned_records = aligned_table[
    aligned_table["alignment_status"] == "matched"
]

# Sort the matched records alphabetically by matrix path
valid_aligned_records = valid_aligned_records.sort_values(
    "matrix_path"
)

# Select the first matched FCM file
target_file_path = Path(
    valid_aligned_records.iloc[0]["matrix_path"]
)

target_file_path

WindowsPath('C:/Users/hiraa/Documents/GitHub/mpi-lemon-connectivity-cognition/data/external/Curvature-FCN-Aging/DATA/FCM/fcm_sub_32301.txt')

## Inspect the raw FCM file format

Before loading the complete file, we inspect the first five lines.

Each line should contain exactly three fields separated by tabs:

1. `roi_i`
2. `roi_j`
3. `correlation`

If the format is incorrect, the validation should stop with an error.

In [7]:
# Read the first five lines from the selected FCM file
with open(target_file_path, "r") as raw_file:
    sample_lines = [next(raw_file) for _ in range(5)]

# Check each sample line
for line in sample_lines:

    # Make sure the line contains tab delimiters
    if "\t" not in line:
        raise ValueError("File is not tab-separated")

    # Split the line into fields using the tab delimiter
    split_fields = line.rstrip("\n").split("\t")

    # Make sure there are exactly three fields
    if len(split_fields) != 3:
        raise ValueError(
            "Line layout does not contain exactly 3 columns"
        )

## Load the FCM edge list and enforce data types

We load the selected FCM file as a tab-separated edge list with no header.

The three columns are assigned the names:

1. `roi_i`
2. `roi_j`
3. `correlation`

We then convert the ROI identifiers to integers and the correlation
values to floating-point numbers.

If any value cannot be converted to the required type, the validation
stops with an error.

In [8]:
# Load the FCM edge list as a tab-separated file with no header
data_table = pd.read_csv(
    target_file_path,
    sep="\t",
    header=None,
    names=["roi_i", "roi_j", "correlation"],
)

# Try to enforce the expected data types
try:

    # Convert the first ROI column to integers
    data_table["roi_i"] = data_table["roi_i"].astype(int)

    # Convert the second ROI column to integers
    data_table["roi_j"] = data_table["roi_j"].astype(int)

    # Convert the correlation column to floating-point numbers
    data_table["correlation"] = data_table["correlation"].astype(float)

except (ValueError, TypeError):
    raise ValueError(
        "Data type format mismatch or unparseable text values detected"
    )
data_table.head()

,roi_i,roi_j,correlation
0,9,109,0.911632
1,19,121,0.906548
2,9,108,0.902196
3,44,149,0.885611
4,108,112,0.882266


##  Validate the FCM edge list

We validate the loaded FCM edge list by checking:

- the number of rows
- the range and number of unique nodes
- self-edges and duplicate ROI pairs
- missing values
- whether all correlation values are within the valid range of -1 to 1


In [9]:
# Validate the FCM edge list
def validate_fcm_edge_list(data_table):

    # Combine both ROI columns to examine all nodes
    nodes = pd.concat([
        data_table["roi_i"],
        data_table["roi_j"]
    ])

    # Create unordered ROI pairs
    pairs = data_table.apply(
        lambda row: tuple(sorted([
            row["roi_i"],
            row["roi_j"]
        ])),
        axis=1
    )

    # Store all validation results
    validation_report = {
        # Rows
        "rows": len(data_table),

        # Nodes
        "minimum_node": nodes.min(),
        "maximum_node": nodes.max(),
        "unique_nodes": nodes.nunique(),

        # Pairs
        "self_edges": (
            data_table["roi_i"] == data_table["roi_j"]
        ).sum(),
        "duplicate_pairs": pairs.duplicated().sum(),

        # Missingness
        "missing_values": data_table.isna().sum().sum(),

        # Correlation range
        "minimum_correlation": data_table["correlation"].min(),
        "maximum_correlation": data_table["correlation"].max(),
        "correlations_in_range": data_table[
            "correlation"
        ].between(-1, 1).all(),
    }

    # Return the complete validation report
    return validation_report

# Run all FCM edge-list validation checks
validation_report = validate_fcm_edge_list(data_table)

# Display the validation report
validation_report

{'rows': 19900,
 'minimum_node': np.int64(0),
 'maximum_node': np.int64(199),
 'unique_nodes': 200,
 'self_edges': np.int64(0),
 'duplicate_pairs': np.int64(0),
 'missing_values': np.int64(0),
 'minimum_correlation': np.float64(-0.6617383995276688),
 'maximum_correlation': np.float64(0.9116317725133024),
 'correlations_in_range': np.True_}

## Produce a compact QC summary

We combine the validation results into a concise quality-control summary
for the selected FCM edge-list file.

The summary records the selected file, its number of rows and nodes,
pair-level checks, missingness, and correlation range.

In [10]:
# Create a compact quality-control summary
qc_summary = {
    "file": target_file_path.name,
    "rows": validation_report["rows"],
    "nodes": validation_report["unique_nodes"],
    "node_range": (
        validation_report["minimum_node"],
        validation_report["maximum_node"]
    ),
    "self_edges": validation_report["self_edges"],
    "duplicate_pairs": validation_report["duplicate_pairs"],
    "missing_values": validation_report["missing_values"],
    "correlation_range": (
        validation_report["minimum_correlation"],
        validation_report["maximum_correlation"]
    ),
    "correlations_in_range": validation_report[
        "correlations_in_range"
    ],
}

qc_summary

{'file': 'fcm_sub_32301.txt',
 'rows': 19900,
 'nodes': 200,
 'node_range': (np.int64(0), np.int64(199)),
 'self_edges': np.int64(0),
 'duplicate_pairs': np.int64(0),
 'missing_values': np.int64(0),
 'correlation_range': (np.float64(-0.6617383995276688),
  np.float64(0.9116317725133024)),
 'correlations_in_range': np.True_}

## Biological interpretation of one FCM edge

Each row in the FCM edge list represents a functional connection between
two brain regions.

For example, an edge connecting `roi_i = 12` and `roi_j = 57` with a
correlation of `0.42` means that the resting-state fMRI signals from
these two regions have a Pearson correlation of 0.42.

A positive correlation indicates that the two regional signals tend to
vary together, while a negative correlation indicates an inverse
relationship.

This represents functional/statistical connectivity and does not by
itself imply a direct anatomical connection or causal relationship.

## Final validation status

The FCM file passes validation only if all required quality-control
checks pass.

The final status is determined automatically from the validation report.

In [11]:
# 1. Determine whether the atlas is 0-indexed (0 to 199) or 1-indexed (1 to 200)
valid_node_range = (
    (validation_report["minimum_node"] == 0 and validation_report["maximum_node"] == 199) or
    (validation_report["minimum_node"] == 1 and validation_report["maximum_node"] == 200)
)

# 2. Determine whether the FCM file passes ALL physical and mathematical checks
validation_passed = (
    validation_report["rows"] == 19900                
    and valid_node_range                                
    and validation_report["self_edges"] == 0            
    and validation_report["duplicate_pairs"] == 0       
    and validation_report["missing_values"] == 0        
    and validation_report["correlations_in_range"])      

# 3. Store and display the final status
validation_status = {
    "file": target_file_path.name,
    "validation_passed": validation_passed,
}

validation_status

{'file': 'fcm_sub_32301.txt', 'validation_passed': np.True_}